# Task 1: Subscription Funnel Analysis

Goal: understand how users move from first app open through to a completed subscription, where they drop off, and what behavioural / segment patterns explain the drop-offs.

Data:
- `app_open_data.csv` — one row per app-open event (ISO timestamps)
- `event_data.csv` — funnel events (epoch-ms timestamps)

The `mycol` column in `app_open_data` is ignored per the task spec.

## Step 0 — Setup & Data Loading

In [231]:
import pandas as pd
import numpy as np
import os
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
from scipy import stats
from scipy.stats import chi2_contingency, fisher_exact
pio.templates.default = 'plotly_white'

import itertools
from IPython.display import Markdown, display
from statsmodels.stats.proportion import proportion_effectsize
from statsmodels.stats.power import zt_ind_solve_power

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 160)

DATA_DIR = 'Growth Task Data'


In [232]:
# Load app_open_data — ISO timestamps
app_open = pd.read_csv(
    f'{DATA_DIR}/app_open_data.csv',
    parse_dates=['event_date', 'timestamp'],
)

# Drop mycol (task says ignore it)
if 'mycol' in app_open.columns:
    app_open = app_open.drop(columns=['mycol'])

print('app_open shape:', app_open.shape)
app_open.head()

app_open shape: (39213, 8)


,event_date,timestamp,device_skey,session_skey,user_skey,is_first_app_open,platform,country_code
0,2023-11-13,2023-11-13 13:15:04.155000+00:00,-1160721689826509359,-6069694996621106477,-4198274634774068263,False,android,vn
1,2023-11-13,2023-11-13 11:32:23.553000+00:00,-4659317238696778303,2034623284200851906,-8645779688131645005,False,android,vn
2,2023-11-13,2023-11-13 02:00:57.992000+00:00,8574209580357038155,7700760069378643919,3858142552250413010,False,android,br
3,2023-11-13,2023-11-13 09:39:29.349000+00:00,-4107400260788282015,-3321024403238710334,3858142552250413010,False,android,br
4,2023-11-13,2023-11-13 07:45:28.949000+00:00,-6656369459599123810,-4504039664568204955,8414965012823618631,False,apple,vn


In [233]:
# Load event_data — epoch-ms timestamps
events = pd.read_csv(f'{DATA_DIR}/event_data.csv')

# Convert epoch-ms -> datetime (UTC)
events['timestamp'] = pd.to_datetime(events['timestamp'], unit='ms', utc=True)

print('events shape:', events.shape)
events.head()

events shape: (151947, 9)


,event_name,timestamp,event_skey,device_skey,user_skey,session_skey,platform,source,country_code
0,registration_open,2023-11-12 09:49:57.828000+00:00,2040623284347280262,-4243964682902975820,NaN,2.275135e+18,android,user_profile,vn
1,registration_open,2023-11-10 15:40:36.825000+00:00,-2687516005135594704,4342432628889352902,NaN,-4.527749e+18,apple,app_start,de
2,registration_open,2023-11-11 13:15:49.187000+00:00,-4765262403246217267,6708321608618278911,NaN,-2.576991e+18,apple,app_start,eg
3,registration_open,2023-11-11 13:51:42.104000+00:00,-2105604357975282211,-3303323474515376120,NaN,8.616913e+18,android,user_profile,br
4,registration_open,2023-11-15 16:08:56.040000+00:00,4905416007576559228,1875496956373071881,NaN,6.956769e+18,android,app_start,eg


In [234]:
# Quick sanity checks — app_open
print('=== app_open.info() ===')
app_open.info()
print('\n=== app_open.describe(include="all") ===')
app_open.describe(include='all').T

=== app_open.info() ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 39213 entries, 0 to 39212
Data columns (total 8 columns):
 #   Column             Non-Null Count  Dtype              
---  ------             --------------  -----              
 0   event_date         39213 non-null  datetime64[ns]     
 1   timestamp          39213 non-null  datetime64[ns, UTC]
 2   device_skey        39213 non-null  int64              
 3   session_skey       39213 non-null  int64              
 4   user_skey          39213 non-null  int64              
 5   is_first_app_open  39213 non-null  bool               
 6   platform           39213 non-null  object             
 7   country_code       39213 non-null  object             
dtypes: bool(1), datetime64[ns, UTC](1), datetime64[ns](1), int64(3), object(2)
memory usage: 2.1+ MB

=== app_open.describe(include="all") ===


,count,unique,top,freq,mean,min,25%,50%,75%,max,std
event_date,39213,NaN,NaN,NaN,2023-11-12 14:01:34.193252352,2023-11-06 00:00:00,2023-11-09 00:00:00,2023-11-13 00:00:00,2023-11-16 00:00:00,2023-11-19 00:00:00,NaN
timestamp,39213,NaN,NaN,NaN,2023-11-13 03:12:40.913421312+00:00,2023-11-06 00:00:26.839000+00:00,2023-11-09 16:51:27.625999872+00:00,2023-11-13 00:35:57.384000+00:00,2023-11-16 15:31:47.217999872+00:00,2023-11-19 23:58:55.728000+00:00,NaN
device_skey,39213.0,NaN,NaN,NaN,-3972625275697125.5,-9223103353847726080.0,-4617330330595899392.0,20344982929111940.0,4623985659542712320.0,9223270975588658176.0,5329070305281298432.0
session_skey,39213.0,NaN,NaN,NaN,-52842394753211208.0,-9223309421273945088.0,-4650920831396664320.0,-36302376267543248.0,4523145656147909632.0,9223042429828378624.0,5315502774495300608.0
user_skey,39213.0,NaN,NaN,NaN,900484451955320192.0,-9223319627764317184.0,-2857192007648710656.0,2288165711503539712.0,3858142552250413056.0,9223191443656040448.0,4810316398691837952.0
is_first_app_open,39213,2,False,38354,NaN,NaN,NaN,NaN,NaN,NaN,NaN
platform,39213,2,android,25143,NaN,NaN,NaN,NaN,NaN,NaN,NaN
country_code,39213,4,br,12942,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [235]:
# Quick sanity checks — events
print('=== events.info() ===')
events.info()
print('\n=== events.describe(include="all") ===')
events.describe(include='all')

=== events.info() ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 151947 entries, 0 to 151946
Data columns (total 9 columns):
 #   Column        Non-Null Count   Dtype              
---  ------        --------------   -----              
 0   event_name    151947 non-null  object             
 1   timestamp     151947 non-null  datetime64[ns, UTC]
 2   event_skey    151947 non-null  int64              
 3   device_skey   151947 non-null  int64              
 4   user_skey     113649 non-null  float64            
 5   session_skey  150752 non-null  float64            
 6   platform      151947 non-null  object             
 7   source        146479 non-null  object             
 8   country_code  151947 non-null  object             
dtypes: datetime64[ns, UTC](1), float64(2), int64(2), object(4)
memory usage: 10.4+ MB

=== events.describe(include="all") ===


,event_name,timestamp,event_skey,device_skey,user_skey,session_skey,platform,source,country_code
count,151947,151947,1.519470e+05,1.519470e+05,1.136490e+05,1.507520e+05,151947,146479,151947
unique,7,NaN,NaN,NaN,NaN,NaN,2,175,4
top,editor_open,NaN,NaN,NaN,NaN,NaN,android,create_flow,br
freq,51412,NaN,NaN,NaN,NaN,NaN,91634,22531,49595
mean,NaN,2023-11-13 04:03:18.267027456+00:00,1.668290e+16,2.693027e+16,3.985468e+16,2.775815e+16,NaN,NaN,NaN
min,NaN,2023-11-06 00:00:58.030000+00:00,-9.223271e+18,-9.223170e+18,-9.223149e+18,-9.223140e+18,NaN,NaN,NaN
25%,NaN,2023-11-09 16:35:34.992499968+00:00,-4.594983e+18,-4.542838e+18,-4.493193e+18,-4.569524e+18,NaN,NaN,NaN
50%,NaN,2023-11-13 02:34:39.328999936+00:00,9.976493e+15,2.961900e+16,1.116500e+16,3.717372e+16,NaN,NaN,NaN
75%,NaN,2023-11-16 16:00:11.027999744+00:00,4.645338e+18,4.644343e+18,4.608453e+18,4.610336e+18,NaN,NaN,NaN
max,NaN,2023-11-19 23:59:50.742000+00:00,9.223308e+18,9.223095e+18,9.223338e+18,9.222286e+18,NaN,NaN,NaN


In [236]:
# Date range covered by each dataset
print('app_open range:', app_open['timestamp'].min(), '→', app_open['timestamp'].max())
print('events   range:', events['timestamp'].min(), '→', events['timestamp'].max())

# Event-name breakdown — confirms the funnel stages we expect
print('\nEvent counts:')
print(events['event_name'].value_counts())

app_open range: 2023-11-06 00:00:26.839000+00:00 → 2023-11-19 23:58:55.728000+00:00
events   range: 2023-11-06 00:00:58.030000+00:00 → 2023-11-19 23:59:50.742000+00:00

Event counts:
event_name
editor_open                51412
object_export              43392
create_flow_open           29342
subscription_offer_open    19223
registration_open           5715
registration_done           2764
subscription_done             99
Name: count, dtype: int64


## Step 1 — Data Cleaning & Preparation

Plan:
1. Audit nulls across both tables.
2. **Primary unit of analysis = `device_skey`** — it's present in every row and tracks the user across pre- and post-registration events. `user_skey` is ~99% null on `registration_open` (user doesn't exist yet) and ~17–23% null on downstream events, so it's unreliable as a primary key.
3. `session_skey` nulls (~1.2k rows) — inspect, then drop. Can't do session-depth analysis without them, and the count is small relative to the dataset.
4. `source` nulls (~5.5k rows) — label as `'unknown'` so they still appear in source breakdowns rather than silently vanishing.
5. Build a device-level attribute table from `app_open` (`is_first_app_open`, platform, country, first-seen timestamp) and attach it to `events` for segmented funnels in Step 3 and new-vs-returning analysis in Step 5b.

In [237]:
# --- 1a. Null audit ---
print('app_open nulls:')
print(app_open.isna().sum())
print('\nevents nulls:')
print(events.isna().sum())

app_open nulls:
event_date           0
timestamp            0
device_skey          0
session_skey         0
user_skey            0
is_first_app_open    0
platform             0
country_code         0
dtype: int64

events nulls:
event_name          0
timestamp           0
event_skey          0
device_skey         0
user_skey       38298
session_skey     1195
platform            0
source           5468
country_code        0
dtype: int64


In [238]:
# --- 1b. Null patterns by event_name ---
# Confirms: user_skey nulls concentrated in registration_open (user doesn't exist yet),
# and source nulls concentrated in specific stages.
null_by_event = (
    events.assign(
        user_skey_null=events['user_skey'].isna(),
        session_skey_null=events['session_skey'].isna(),
        source_null=events['source'].isna(),
    )
    .groupby('event_name')
    .agg(
        rows=('event_name', 'size'),
        user_skey_null=('user_skey_null', 'sum'),
        session_skey_null=('session_skey_null', 'sum'),
        source_null=('source_null', 'sum'),
    )
)
for c in ['user_skey_null', 'session_skey_null', 'source_null']:
    null_by_event[f'{c}_pct'] = (null_by_event[c] / null_by_event['rows'] * 100).round(1)
null_by_event

,rows,user_skey_null,session_skey_null,source_null,user_skey_null_pct,session_skey_null_pct,source_null_pct
event_name,,,,,,,
create_flow_open,29342,6188,0,214,21.1,0.0,0.7
editor_open,51412,11930,1195,0,23.2,2.3,0.0
object_export,43392,9830,0,2403,22.7,0.0,5.5
registration_done,2764,1290,0,2764,46.7,0.0,100.0
registration_open,5715,5661,0,85,99.1,0.0,1.5
subscription_done,99,13,0,0,13.1,0.0,0.0
subscription_offer_open,19223,3386,0,2,17.6,0.0,0.0


In [239]:
# --- 1c. Drop rows with missing session_skey ---
# Small count relative to the dataset, and session-depth analysis (Step 5c) needs it.
# Safer to drop than impute a session ID we'd be inventing.
rows_before = len(events)
events = events.dropna(subset=['session_skey']).copy()
print(f'Dropped {rows_before - len(events):,} rows with null session_skey.')
print(f'Remaining events: {len(events):,}')

Dropped 1,195 rows with null session_skey.
Remaining events: 150,752


In [240]:
# --- 1d. Flag source nulls as 'unknown' ---
# Keep these rows so source breakdowns still reflect the full population.
events['source'] = events['source'].fillna('unknown')
print(events['source'].value_counts(dropna=False).head(15))

source
create_flow           22015
my_network            20201
photo                 17393
editor_screen         16862
photo_choose           9288
autostart              6658
editor_add_objects     6228
share_screen           5586
unknown                5468
editor_effects         4147
file_manager           3346
app_start              3033
export_screen          1911
state_recover          1692
editor_square_fit      1586
Name: count, dtype: int64


In [241]:
# --- 1e. Normalize ID dtypes so joins behave ---
for col in ['device_skey', 'session_skey', 'user_skey']:
    if col in events.columns:
        events[col] = events[col].astype('Int64')  # nullable int
    if col in app_open.columns:
        app_open[col] = app_open[col].astype('Int64')

print('events ID dtypes:')
print(events[['device_skey', 'session_skey', 'user_skey']].dtypes)
print('\napp_open ID dtypes:')
print(app_open[['device_skey', 'session_skey', 'user_skey']].dtypes)

events ID dtypes:
device_skey     Int64
session_skey    Int64
user_skey       Int64
dtype: object

app_open ID dtypes:
device_skey     Int64
session_skey    Int64
user_skey       Int64
dtype: object


In [242]:
# --- 1f. Device-level attribute table ---
# One row per device with:
#   - is_first_app_open: True if the device was a first-time install during the window
#   - platform / country_code: from the earliest app_open record
#   - first_seen / last_seen / app_open_count: basic engagement stats
app_open_sorted = app_open.sort_values('timestamp')

device_attrs = (
    app_open_sorted
    .groupby('device_skey')
    .agg(
        is_first_app_open=('is_first_app_open', 'max'),
        platform=('platform', 'first'),
        country_code=('country_code', 'first'),
        first_seen=('timestamp', 'min'),
        last_seen=('timestamp', 'max'),
        app_open_count=('timestamp', 'size'),
    )
    .reset_index()
)
print('device_attrs shape:', device_attrs.shape)
device_attrs.head()

device_attrs shape: (38730, 7)


,device_skey,is_first_app_open,platform,country_code,first_seen,last_seen,app_open_count
0,-9223103353847725703,False,android,br,2023-11-10 20:00:25.742000+00:00,2023-11-10 20:00:25.742000+00:00,1
1,-9222863142111266937,False,apple,de,2023-11-06 20:47:17.974000+00:00,2023-11-06 20:47:17.974000+00:00,1
2,-9222607425128578177,False,apple,br,2023-11-09 22:03:07.490000+00:00,2023-11-09 22:03:07.490000+00:00,1
3,-9221898489869906465,False,android,br,2023-11-17 15:55:11.046000+00:00,2023-11-17 15:55:11.046000+00:00,1
4,-9221248224369718564,False,android,vn,2023-11-18 08:30:35.680000+00:00,2023-11-18 08:30:35.680000+00:00,1


In [243]:
# --- 1g. Attach device attributes to events ---
# Left-join on device_skey so we keep every event. Events whose device never
# appeared in app_open get NaN attrs — we'll quantify that next.
events_enriched = events.merge(
    device_attrs[['device_skey', 'is_first_app_open', 'first_seen', 'app_open_count']],
    on='device_skey',
    how='left',
)

missing_attrs = events_enriched['is_first_app_open'].isna().sum()
total_rows = len(events_enriched)
missing_pct = missing_attrs / total_rows * 100
print(f'Events with no matching device in app_open: {missing_attrs:,} of {total_rows:,} ({missing_pct:.1f}%)')
events_enriched.head()

Events with no matching device in app_open: 145,087 of 150,752 (96.2%)


,event_name,timestamp,event_skey,device_skey,user_skey,session_skey,platform,source,country_code,is_first_app_open,first_seen,app_open_count
0,registration_open,2023-11-12 09:49:57.828000+00:00,2040623284347280262,-4243964682902975820,<NA>,2275134884222651648,android,user_profile,vn,NaN,NaT,NaN
1,registration_open,2023-11-10 15:40:36.825000+00:00,-2687516005135594704,4342432628889352902,<NA>,-4527749219244896256,apple,app_start,de,NaN,NaT,NaN
2,registration_open,2023-11-11 13:15:49.187000+00:00,-4765262403246217267,6708321608618278911,<NA>,-2576991296053508608,apple,app_start,eg,NaN,NaT,NaN
3,registration_open,2023-11-11 13:51:42.104000+00:00,-2105604357975282211,-3303323474515376120,<NA>,8616913279251193856,android,user_profile,br,NaN,NaT,NaN
4,registration_open,2023-11-15 16:08:56.040000+00:00,4905416007576559228,1875496956373071881,<NA>,6956768589414704128,android,app_start,eg,NaN,NaT,NaN


In [244]:
# --- 1h. Summary of cleaned state ---
print('CLEANED STATE')
print('-' * 40)
print(f'app_open rows             : {len(app_open):,}')
print(f'unique devices (app_open) : {app_open["device_skey"].nunique():,}')
print(f'events rows               : {len(events):,}')
print(f'unique devices (events)   : {events["device_skey"].nunique():,}')
print(f'device_attrs rows         : {len(device_attrs):,}')
print(f'events_enriched rows      : {len(events_enriched):,}')

CLEANED STATE
----------------------------------------
app_open rows             : 39,213
unique devices (app_open) : 38,730
events rows               : 150,752
unique devices (events)   : 32,183
device_attrs rows         : 38,730
events_enriched rows      : 150,752


## Step 2 — Overall Funnel Construction

Funnel stages (from the task spec), measured as **unique devices** (`device_skey`) that reached each stage:

1. `app_open` — from `app_open_data`
2. `registration_open`
3. `registration_done`
4. `create_flow_open`
5. `editor_open`
6. `subscription_offer_open`
7. `object_export`
8. `subscription_done`

For each stage we compute:
- **stage-to-stage conversion** — `devices_N / devices_{N-1}`
- **overall conversion from top** — `devices_N / devices_app_open`

Note: this is a "ever reached stage" funnel, not strictly sequential — a device is counted at any stage it ever hit during the window. That matches how product teams typically track funnels; a strictly-sequential funnel would over-penalize users who hit events out of order.

In [245]:
# --- 2a. Funnel stages in order ---
FUNNEL_STAGES = [
    'app_open',
    'registration_open',
    'registration_done',
    'create_flow_open',
    'editor_open',
    'subscription_offer_open',
    'object_export',
    'subscription_done',
]

In [246]:
# --- 2b. Unique-device count per stage ---
# Stage 1 comes from the app_open table; stages 2-8 come from events.
event_stage_devices = events.groupby('event_name')['device_skey'].nunique()

stage_counts = {'app_open': app_open['device_skey'].nunique()}
for stage in FUNNEL_STAGES[1:]:
    stage_counts[stage] = int(event_stage_devices.get(stage, 0))

for stage, n in stage_counts.items():
    print(f'{stage:<25} {n:>8,}')

app_open                    38,730
registration_open            4,099
registration_done            2,300
create_flow_open            24,681
editor_open                 25,834
subscription_offer_open     11,451
object_export               20,115
subscription_done               92


In [247]:
# --- 2c. Funnel table: stage-to-stage + overall conversion ---
funnel = pd.DataFrame({
    'stage': FUNNEL_STAGES,
    'devices': [stage_counts[s] for s in FUNNEL_STAGES],
})
funnel['stage_to_stage_conv_pct'] = (funnel['devices'] / funnel['devices'].shift(1) * 100).round(2)
funnel['overall_conv_from_top_pct'] = (funnel['devices'] / funnel['devices'].iloc[0] * 100).round(3)
funnel

,stage,devices,stage_to_stage_conv_pct,overall_conv_from_top_pct
0,app_open,38730,NaN,100.000
1,registration_open,4099,10.58,10.584
2,registration_done,2300,56.11,5.939
3,create_flow_open,24681,1073.09,63.726
4,editor_open,25834,104.67,66.703
5,subscription_offer_open,11451,44.33,29.566
6,object_export,20115,175.66,51.936
7,subscription_done,92,0.46,0.238


In [248]:
# --- 2d. Funnel chart (plotly) ---
fig = go.Figure(go.Funnel(
    y=funnel['stage'],
    x=funnel['devices'],
    textposition='inside',
    textinfo='value+percent initial+percent previous',
    marker={'color': "#1245DE"},
))
fig.update_layout(
    title='Overall Subscription Funnel — unique devices',
    height=520,
    margin=dict(l=180, r=40, t=60, b=40),
)
fig.show()

In [249]:
# --- 2e. Identify the biggest drop-off ---
drops = pd.DataFrame({
    'from_stage': FUNNEL_STAGES[:-1],
    'to_stage': FUNNEL_STAGES[1:],
    'from_devices': funnel['devices'].iloc[:-1].values,
    'to_devices': funnel['devices'].iloc[1:].values,
})
drops['lost'] = drops['from_devices'] - drops['to_devices']
drops['drop_pct'] = (drops['lost'] / drops['from_devices'] * 100).round(2)

print('Stage-to-stage drops, sorted by absolute devices lost:')
drops_by_absolute = drops.sort_values('lost', ascending=False).reset_index(drop=True)
print(drops_by_absolute.to_string(index=False))

print('\nStage-to-stage drops, sorted by % drop:')
drops_by_pct = drops.sort_values('drop_pct', ascending=False).reset_index(drop=True)
print(drops_by_pct.to_string(index=False))

biggest = drops_by_absolute.iloc[0]
print(f"\n>> Biggest absolute drop: {biggest['from_stage']} -> {biggest['to_stage']}  "
      f"({biggest['lost']:,} devices, {biggest['drop_pct']}%)")

Stage-to-stage drops, sorted by absolute devices lost:
             from_stage                to_stage  from_devices  to_devices   lost  drop_pct
               app_open       registration_open         38730        4099  34631     89.42
          object_export       subscription_done         20115          92  20023     99.54
            editor_open subscription_offer_open         25834       11451  14383     55.67
      registration_open       registration_done          4099        2300   1799     43.89
       create_flow_open             editor_open         24681       25834  -1153     -4.67
subscription_offer_open           object_export         11451       20115  -8664    -75.66
      registration_done        create_flow_open          2300       24681 -22381   -973.09

Stage-to-stage drops, sorted by % drop:
             from_stage                to_stage  from_devices  to_devices   lost  drop_pct
          object_export       subscription_done         20115          92  20023     

## Step 3 — Segmented Funnels

Segment the funnel by **platform** (android / apple) and **country_code** (br / vn / eg / de) to surface where conversion concentrates. We reuse the same "unique device per stage" logic as Step 2.

For each segment we report:
- device counts at every funnel stage
- % of that segment's `app_open` users reaching each downstream stage
- **headline metric**: `subscription_done / app_open` = the full-funnel conversion rate

Platform/country is taken from each row's own `platform` / `country_code` column (present in both tables), which matches how the task spec frames the segmentation.

In [250]:
# --- 3a. Helper: given a subset of app_open and events, return per-stage unique-device counts ---
def compute_funnel_counts(app_open_subset, events_subset):
    counts = {'app_open': app_open_subset['device_skey'].nunique()}
    stage_counts = events_subset.groupby('event_name')['device_skey'].nunique()
    for stage in FUNNEL_STAGES[1:]:
        counts[stage] = int(stage_counts.get(stage, 0))
    return counts


def build_segment_funnel(app_open_df, events_df, segment_col):
    """Return (counts_df, conv_from_top_df). Rows are FUNNEL_STAGES, cols are segment values."""
    segments = sorted(app_open_df[segment_col].dropna().unique())
    counts = {}
    for seg in segments:
        ao = app_open_df[app_open_df[segment_col] == seg]
        ev = events_df[events_df[segment_col] == seg]
        counts[seg] = compute_funnel_counts(ao, ev)
    counts_df = pd.DataFrame(counts).reindex(FUNNEL_STAGES)
    conv_df = (counts_df.div(counts_df.iloc[0]) * 100).round(3)
    return counts_df, conv_df

In [251]:
# --- 3b. Funnel by platform ---
platform_counts, platform_conv = build_segment_funnel(app_open, events, 'platform')

print('Device counts per stage:')
print(platform_counts)
print('\n% of app_open reaching each stage:')
print(platform_conv)

print('\nHeadline — full-funnel conversion (subscription_done of app_open):')
for p in platform_counts.columns:
    ao = platform_counts.loc['app_open', p]
    sd = platform_counts.loc['subscription_done', p]
    print(f'  {p:<8} {sd:>3,} of {ao:>8,}  =  {sd / ao * 100:.3f}%')

Device counts per stage:
                         android  apple
app_open                   24766  13964
registration_open           3073   1026
registration_done           1566    734
create_flow_open           14480  10201
editor_open                15700  10134
subscription_offer_open     5560   5891
object_export              12321   7794
subscription_done             33     59

% of app_open reaching each stage:
                         android    apple
app_open                 100.000  100.000
registration_open         12.408    7.347
registration_done          6.323    5.256
create_flow_open          58.467   73.052
editor_open               63.393   72.572
subscription_offer_open   22.450   42.187
object_export             49.750   55.815
subscription_done          0.133    0.423

Headline — full-funnel conversion (subscription_done of app_open):
  android   33 of   24,766  =  0.133%
  apple     59 of   13,964  =  0.423%


In [252]:
# --- 3c. Side-by-side funnel chart: platform ---
fig = go.Figure()
for p in platform_counts.columns:
    fig.add_trace(go.Funnel(
        name=p,
        y=FUNNEL_STAGES,
        x=platform_counts[p].values,
        textinfo='value+percent initial',
    ))
fig.update_layout(
    title='Subscription Funnel by Platform',
    height=560,
    margin=dict(l=180, r=40, t=60, b=40),
    legend_title_text='platform',
)
fig.show()

In [253]:
# --- 3d. Grouped bar: % reaching each stage, by platform ---
conv_long = (
    platform_conv
    .reset_index()
    .rename(columns={'index': 'stage'})
    .melt(id_vars='stage', var_name='platform', value_name='pct_of_app_open')
)
fig = px.bar(
    conv_long,
    x='stage', y='pct_of_app_open',
    color='platform', barmode='group',
    category_orders={'stage': FUNNEL_STAGES},
    title='% of app_open-ers reaching each funnel stage, by platform',
)
fig.update_layout(xaxis_tickangle=-30, yaxis_title='% of platform\'s app_open users')
fig.show()

In [254]:
# --- 3e. Funnel by country ---
country_counts, country_conv = build_segment_funnel(app_open, events, 'country_code')

print('Device counts per stage:')
print(country_counts)
print('\n% of app_open reaching each stage:')
print(country_conv)

print('\nHeadline — full-funnel conversion (subscription_done of app_open):')
for c in country_counts.columns:
    ao = country_counts.loc['app_open', c]
    sd = country_counts.loc['subscription_done', c]
    rate = sd / ao * 100 if ao else 0
    print(f'  {c:<4} {sd:>3,} of {ao:>8,}  =  {rate:.3f}%')

Device counts per stage:
                            br    de    eg     vn
app_open                 12785  4202  9960  11786
registration_open         1900   385   968    847
registration_done          942   205   513    640
create_flow_open          7664  2641  5896   8482
editor_open               8106  2818  6617   8294
subscription_offer_open   3790   602  2141   4918
object_export             6349  2172  5130   6466
subscription_done           45    15     9     23

% of app_open reaching each stage:
                              br       de       eg       vn
app_open                 100.000  100.000  100.000  100.000
registration_open         14.861    9.162    9.719    7.186
registration_done          7.368    4.879    5.151    5.430
create_flow_open          59.945   62.851   59.197   71.967
editor_open               63.402   67.063   66.436   70.372
subscription_offer_open   29.644   14.327   21.496   41.727
object_export             49.660   51.690   51.506   54.862
subscript

In [255]:
# --- 3f. Side-by-side funnel chart: country ---
fig = go.Figure()
for c in country_counts.columns:
    fig.add_trace(go.Funnel(
        name=c,
        y=FUNNEL_STAGES,
        x=country_counts[c].values,
        textinfo='value+percent initial',
    ))
fig.update_layout(
    title='Subscription Funnel by Country',
    height=560,
    margin=dict(l=180, r=40, t=60, b=40),
    legend_title_text='country',
)
fig.show()

In [256]:
# --- 3g. Grouped bar: % reaching each stage, by country ---
conv_long_c = (
    country_conv
    .reset_index()
    .rename(columns={'index': 'stage'})
    .melt(id_vars='stage', var_name='country', value_name='pct_of_app_open')
)
fig = px.bar(
    conv_long_c,
    x='stage', y='pct_of_app_open',
    color='country', barmode='group',
    category_orders={'stage': FUNNEL_STAGES},
    title='% of app_open-ers reaching each funnel stage, by country',
)
fig.update_layout(xaxis_tickangle=-30, yaxis_title='% of country\'s app_open users')
fig.show()

In [257]:
# --- 3h. Ranked segment-conversion summary (for Step 7 recs) ---
def segment_summary(counts_df, segment_name):
    rows = []
    for seg in counts_df.columns:
        ao = counts_df.loc['app_open', seg]
        sd = counts_df.loc['subscription_done', seg]
        rows.append({
            segment_name: seg,
            'app_open_devices': int(ao),
            'subscription_done_devices': int(sd),
            'conv_pct': round(sd / ao * 100, 3) if ao else 0.0,
        })
    return pd.DataFrame(rows).sort_values('conv_pct', ascending=False).reset_index(drop=True)

print('Platforms ranked by end-to-end conversion:')
print(segment_summary(platform_counts, 'platform').to_string(index=False))

print('\nCountries ranked by end-to-end conversion:')
print(segment_summary(country_counts, 'country').to_string(index=False))

Platforms ranked by end-to-end conversion:
platform  app_open_devices  subscription_done_devices  conv_pct
   apple             13964                         59     0.423
 android             24766                         33     0.133

Countries ranked by end-to-end conversion:
country  app_open_devices  subscription_done_devices  conv_pct
     de              4202                         15     0.357
     br             12785                         45     0.352
     vn             11786                         23     0.195
     eg              9960                          9     0.090


## Step 4 — Registration Deep Dive

Registration is typically a top drop-off point. Here we zoom in on the `registration_open → registration_done` step at device granularity:

- **Definition**: a device "completes registration" if we observe *both* a `registration_open` and a `registration_done` for that device.
- **Completion rate = |openers ∩ completers| / |openers|**
- Break it down by platform and country, then cross them.
- Also report devices with `registration_done` but no logged `registration_open` — a data-quality signal (missed event, or registration entered via a path that doesn't fire the open event).

In [258]:
# --- 4a. Overall registration completion (device-level) ---
reg_events = events[events['event_name'].isin(['registration_open', 'registration_done'])]

reg_open_devices = set(events.loc[events['event_name'] == 'registration_open', 'device_skey'].dropna())
reg_done_devices = set(events.loc[events['event_name'] == 'registration_done', 'device_skey'].dropna())

both = reg_open_devices & reg_done_devices
only_done = reg_done_devices - reg_open_devices

completion_rate = len(both) / len(reg_open_devices) * 100 if reg_open_devices else 0.0

print(f'Devices with registration_open :          {len(reg_open_devices):,}')
print(f'Devices with registration_done :          {len(reg_done_devices):,}')
print(f'Devices with BOTH events :                {len(both):,}')
print(f'Devices with done but no open logged :    {len(only_done):,}  '
      f'({len(only_done) / len(reg_done_devices) * 100:.1f}% of done-ers)')
print(f'\n>> Device-level completion rate: {completion_rate:.2f}%')

# Row-level sanity check (for comparison with any "raw events ratio")
n_open_rows = (events['event_name'] == 'registration_open').sum()
n_done_rows = (events['event_name'] == 'registration_done').sum()
print(f'\n(For reference: row-level open={n_open_rows:,}, done={n_done_rows:,}, '
      f'raw ratio={n_done_rows / n_open_rows * 100:.2f}% — different metric)')

Devices with registration_open :          4,099
Devices with registration_done :          2,300
Devices with BOTH events :                2,251
Devices with done but no open logged :    49  (2.1% of done-ers)

>> Device-level completion rate: 54.92%

(For reference: row-level open=5,715, done=2,764, raw ratio=48.36% — different metric)


In [259]:
# --- 4b. Helper & segment breakdowns ---
def reg_completion_for(events_subset):
    openers = set(events_subset.loc[events_subset['event_name'] == 'registration_open', 'device_skey'].dropna())
    doners = set(events_subset.loc[events_subset['event_name'] == 'registration_done', 'device_skey'].dropna())
    completed = openers & doners
    return {
        'openers': len(openers),
        'completers': len(completed),
        'completion_pct': round(len(completed) / len(openers) * 100, 2) if openers else 0.0,
    }


def reg_breakdown(segment_col):
    segs = sorted(events[segment_col].dropna().unique())
    rows = {seg: reg_completion_for(events[events[segment_col] == seg]) for seg in segs}
    return (
        pd.DataFrame(rows).T
        .rename_axis(segment_col)
        .reset_index()
        .sort_values('completion_pct', ascending=False)
        .reset_index(drop=True)
    )


platform_reg = reg_breakdown('platform')
country_reg = reg_breakdown('country_code')

print('Registration completion by platform:')
print(platform_reg.to_string(index=False))
print('\nRegistration completion by country:')
print(country_reg.to_string(index=False))

Registration completion by platform:
platform  openers  completers  completion_pct
   apple   1026.0       725.0           70.66
 android   3073.0      1526.0           49.66

Registration completion by country:
country_code  openers  completers  completion_pct
          vn    847.0       632.0           74.62
          de    385.0       200.0           51.95
          eg    968.0       500.0           51.65
          br   1900.0       919.0           48.37


In [260]:
# --- 4c. Platform x country crosstab ---
combo_rows = []
for (p, c), grp in events.groupby(['platform', 'country_code']):
    combo_rows.append({'platform': p, 'country_code': c, **reg_completion_for(grp)})

platform_country_reg = (
    pd.DataFrame(combo_rows)
    .sort_values(['platform', 'completion_pct'], ascending=[True, False])
    .reset_index(drop=True)
)
print('Registration completion, platform × country:')
print(platform_country_reg.to_string(index=False))

# Pivot view for quick reading
pivot = platform_country_reg.pivot(index='country_code', columns='platform', values='completion_pct')
print('\nCompletion % pivot (rows=country, cols=platform):')
print(pivot)

Registration completion, platform × country:
platform country_code  openers  completers  completion_pct
 android           vn      512         369           72.07
 android           eg      864         440           50.93
 android           de      132          65           49.24
 android           br     1566         652           41.63
   apple           br      334         267           79.94
   apple           vn      335         263           78.51
   apple           eg      104          60           57.69
   apple           de      253         135           53.36

Completion % pivot (rows=country, cols=platform):
platform      android  apple
country_code                
br              41.63  79.94
de              49.24  53.36
eg              50.93  57.69
vn              72.07  78.51


In [261]:
# --- 4d. Visualizations ---
fig = px.bar(
    platform_reg, x='platform', y='completion_pct',
    text='completion_pct',
    title='Registration completion rate by platform',
    hover_data=['openers', 'completers'],
)
fig.update_traces(texttemplate='%{text:.1f}%', textposition='outside')
fig.update_layout(yaxis_title='completion %', yaxis_range=[0, 100])
fig.show()

fig = px.bar(
    country_reg, x='country_code', y='completion_pct',
    text='completion_pct',
    title='Registration completion rate by country',
    hover_data=['openers', 'completers'],
)
fig.update_traces(texttemplate='%{text:.1f}%', textposition='outside')
fig.update_layout(yaxis_title='completion %', yaxis_range=[0, 100])
fig.show()

# Heatmap: country x platform
fig = px.imshow(
    pivot, text_auto='.1f', color_continuous_scale='Blues',
    aspect='auto',
    title='Registration completion % — country × platform',
    labels=dict(color='completion %'),
)
fig.show()

### Hypotheses to revisit in Step 7

Things to check against the numbers above before writing them up as findings:

- **Sign-in method friction**: if Apple completes much higher than Android, "Sign in with Apple" (one-tap) may be converting better than whatever primary method Android offers. Worth confirming if `source` on registration_done carries the auth provider.
- **Country-level UX**: a low completer rate in a high-traffic country (e.g. VN or BR) is a concentrated opportunity — one UX change hits a big audience.
- **"done without open logged"**: if this fraction is non-trivial (>5%), either an entry path bypasses the open screen (auto-login? deep link from email verification?) or we're under-firing the `registration_open` event. Both are worth flagging to product/analytics.
- **Form friction vs abandonment**: we don't have field-level telemetry here, so we can only measure the endpoints. A follow-up would be to instrument per-field events to identify *where* inside the form users drop.

## Step 5 — Behavioral Analysis

Three sub-analyses:

- **5a — Subscription trigger**: which `source` screens surface the paywall, and which convert best.
- **5b — New vs returning users**: does being a first-time installer meaningfully change the funnel?
- **5c — Session depth before the paywall**: do users who've done more in a session convert at higher rates — is there a "sweet spot" for paywall timing?

Unit of analysis: `(device_skey, session_skey)` pairs within the events table. A "converting session" is one where a `subscription_done` event appears in the same session.

### 5a — Subscription trigger analysis

For `subscription_offer_open`, the `source` field tells us which in-app screen triggered the paywall. For each source, we want:

1. **Volume** — how many offer views come from this source (raw events and unique offer-sessions).
2. **Conversion** — of the sessions where a paywall was shown with this source, what fraction also had a `subscription_done`.

Attribution rule: a paywall session "converts" if the same `(device_skey, session_skey)` pair contains a `subscription_done` event. Raw-event volume and session-level volume are both reported so we can tell whether a big source number is being inflated by many repeat views per session.

In [262]:
# --- 5a.I  Extract offer and subscription events ---
offers = events[events['event_name'] == 'subscription_offer_open'].copy()
dones = events[events['event_name'] == 'subscription_done'].copy()

print(f'subscription_offer_open rows :   {len(offers):,}')
print(f'subscription_done rows :         {len(dones):,}')
print(f'unique offer sessions :          {offers[["device_skey","session_skey"]].drop_duplicates().shape[0]:,}')
print(f'unique done sessions :           {dones[["device_skey","session_skey"]].drop_duplicates().shape[0]:,}')

# Raw volume by source (every offer view counts)
print('\nRaw offer views by source:')
print(offers['source'].value_counts().head(15))

subscription_offer_open rows :   19,223
subscription_done rows :         99
unique offer sessions :          11,483
unique done sessions :           92

Raw offer views by source:
source
photo_choose              9280
editor_complete           1229
editor_add_text           1113
sign_in                    815
picsart_upload             811
registration_skip          743
collage_photo_choose       710
editor_export_screen       692
editor_main_toolbar        535
registration               494
editor_add_photo           378
tool_remove_bg             301
editor_effect_apply        298
editor_add_sticker         183
video_editor_watermark     183
Name: count, dtype: int64


In [263]:
# --- 5a.II  Source-level conversion (session-based attribution) ---
# Converting sessions: set of (device_skey, session_skey) that contain a subscription_done
converting_sessions = set(
    map(tuple, dones[['device_skey', 'session_skey']].dropna().values.tolist())
)

# Each row below is one (device, session, source) tuple that saw the paywall.
# We de-dup so a session that saw 10 paywall views with the same source counts once.
offer_sessions = (
    offers[['device_skey', 'session_skey', 'source']]
    .dropna(subset=['device_skey', 'session_skey'])
    .drop_duplicates()
    .copy()
)
offer_sessions['converted'] = [
    (d, s) in converting_sessions
    for d, s in zip(offer_sessions['device_skey'], offer_sessions['session_skey'])
]

source_conv = (
    offer_sessions.groupby('source')
    .agg(offer_sessions=('converted', 'size'),
         converting_sessions=('converted', 'sum'))
    .reset_index()
)
source_conv['conv_rate_pct'] = (source_conv['converting_sessions'] /
                                source_conv['offer_sessions'] * 100).round(3)

# Add the raw view count for context
source_conv = source_conv.merge(
    offers['source'].value_counts().rename('raw_offer_views').reset_index().rename(columns={'index': 'source'}),
    on='source', how='left',
)

source_conv = source_conv[['source', 'raw_offer_views', 'offer_sessions',
                           'converting_sessions', 'conv_rate_pct']]
print('Source-level paywall volume & conversion (sorted by conversion rate):')
print(source_conv.sort_values('conv_rate_pct', ascending=False).to_string(index=False))

print('\nSame table, sorted by volume:')
print(source_conv.sort_values('raw_offer_views', ascending=False).to_string(index=False))

Source-level paywall volume & conversion (sorted by conversion rate):
                             source  raw_offer_views  offer_sessions  converting_sessions  conv_rate_pct
                          ai_writer                2               2                    1         50.000
            history_player_add_text                6               4                    2         50.000
                      collage_apply                9               5                    2         40.000
editor_beautify_reshape_refine_plus                3               3                    1         33.333
              history_player_social               10               3                    1         33.333
            editor_text_custom_font               12               6                    2         33.333
                   collage_add_text              110              26                    4         15.385
                        tool_cutout               10               7                    1 

In [264]:
# --- 5a.III  Volume vs conversion scatter ---
# Shows "big and high-converting" (upper right) vs "big but low-converting" (lower right — the optimization target)
sc = source_conv.copy()
fig = px.scatter(
    sc, x='offer_sessions', y='conv_rate_pct',
    size='raw_offer_views', text='source',
    hover_data=['raw_offer_views', 'converting_sessions'],
    title='Paywall source: volume vs. conversion rate',
    labels={'offer_sessions': 'unique offer-sessions', 'conv_rate_pct': 'conversion rate (%)'},
)
fig.update_traces(textposition='top center')
fig.update_layout(height=500)
fig.show()

# Complementary bar: top sources by conversion (filter to sources with enough volume)
MIN_SESSIONS = 50  # adjust if too strict; avoids 1-off noise
top_by_rate = (sc[sc['offer_sessions'] >= MIN_SESSIONS]
               .sort_values('conv_rate_pct', ascending=False).head(15))
fig = px.bar(
    top_by_rate, x='source', y='conv_rate_pct',
    text='conv_rate_pct',
    hover_data=['offer_sessions', 'converting_sessions'],
    title=f'Top sources by paywall → sub conversion rate (min {MIN_SESSIONS} offer sessions)',
)
fig.update_traces(texttemplate='%{text:.2f}%', textposition='outside')
fig.update_layout(xaxis_tickangle=-30, yaxis_title='conversion rate (%)')
fig.show()

### 5b — New vs returning users

We tag each device by `is_first_app_open` (from `device_attrs` built in Step 1f):
- **New**: the device has a `True` first-open row in the window — a fresh install.
- **Returning**: the device appeared in `app_open` but was never flagged as a first open — an already-installed user returning.
- **Unknown**: device has events but never appeared in `app_open` (missing attribution) — reported separately, not used in the new/returning comparison.

For each cohort we rebuild the full funnel and the end-to-end conversion rate.

In [265]:
# --- 5b.I  Cohort sizes ---
new_devices = set(device_attrs.loc[device_attrs['is_first_app_open'] == True, 'device_skey'])
returning_devices = set(device_attrs.loc[device_attrs['is_first_app_open'] == False, 'device_skey'])

events_devices = set(events['device_skey'].dropna())
unknown_devices = events_devices - new_devices - returning_devices

print(f'New install devices (is_first_app_open=True)     : {len(new_devices):,}')
print(f'Returning devices   (in app_open, not first-open): {len(returning_devices):,}')
print(f'Unknown devices     (in events, not in app_open) : {len(unknown_devices):,}')

New install devices (is_first_app_open=True)     : 859
Returning devices   (in app_open, not first-open): 37,871
Unknown devices     (in events, not in app_open) : 30,994


In [266]:
# --- 5b.II  Funnel counts per cohort ---
def cohort_funnel(device_set):
    ao = app_open[app_open['device_skey'].isin(device_set)]
    ev = events[events['device_skey'].isin(device_set)]
    return compute_funnel_counts(ao, ev)


cohort_counts = pd.DataFrame({
    'new_install':   cohort_funnel(new_devices),
    'returning':     cohort_funnel(returning_devices),
}).reindex(FUNNEL_STAGES)

cohort_conv = (cohort_counts.div(cohort_counts.iloc[0]) * 100).round(3)

print('Device counts per stage:')
print(cohort_counts)
print('\n% of cohort reaching each stage:')
print(cohort_conv)

print('\nHeadline — subscription_done of app_open:')
for col in cohort_counts.columns:
    ao = cohort_counts.loc['app_open', col]
    sd = cohort_counts.loc['subscription_done', col]
    print(f'  {col:<12} {sd:>3,} of {ao:>6,}  =  {sd / ao * 100:.3f}%' if ao else f'  {col} — no app_open devices')

Device counts per stage:
                         new_install  returning
app_open                         859      37871
registration_open                 15        106
registration_done                 14         39
create_flow_open                  17        801
editor_open                       13        936
subscription_offer_open           14        338
object_export                      7        759
subscription_done                  0          0

% of cohort reaching each stage:
                         new_install  returning
app_open                     100.000    100.000
registration_open              1.746      0.280
registration_done              1.630      0.103
create_flow_open               1.979      2.115
editor_open                    1.513      2.472
subscription_offer_open        1.630      0.893
object_export                  0.815      2.004
subscription_done              0.000      0.000

Headline — subscription_done of app_open:
  new_install    0 of    859  =  0

In [267]:
# --- 5b.III  Visualize cohort funnels ---
fig = go.Figure()
for col in cohort_counts.columns:
    fig.add_trace(go.Funnel(
        name=col,
        y=FUNNEL_STAGES,
        x=cohort_counts[col].values,
        textinfo='value+percent initial',
    ))
fig.update_layout(
    title='Subscription Funnel: new install vs returning',
    height=560,
    margin=dict(l=180, r=40, t=60, b=40),
    legend_title_text='cohort',
)
fig.show()

conv_long_n = (
    cohort_conv
    .reset_index()
    .rename(columns={'index': 'stage'})
    .melt(id_vars='stage', var_name='cohort', value_name='pct_of_app_open')
)
fig = px.bar(
    conv_long_n,
    x='stage', y='pct_of_app_open',
    color='cohort', barmode='group',
    category_orders={'stage': FUNNEL_STAGES},
    title='% of app_open reaching each stage, by cohort',
)
fig.update_layout(xaxis_tickangle=-30)
fig.show()

### 5c — Session depth & conversion

For each session that saw a paywall (`subscription_offer_open`), count how many events happened in the same session **before** the first offer — this is our proxy for in-session engagement. Then split by whether the session converted (`subscription_done` in the same session) and compare depth distributions.

The product question: *is there a sweet spot of engagement before which the paywall converts poorly, and after which it plateaus?* If yes → delay the paywall until the user has done N actions.

In [268]:
# --- 5c.I  First-offer timestamp per session ---
first_offer = (
    offers.groupby(['device_skey', 'session_skey'], as_index=False)['timestamp']
    .min()
    .rename(columns={'timestamp': 'first_offer_ts'})
)
print(f'Sessions that saw a paywall: {len(first_offer):,}')

# Attach first-offer timestamp back onto events; keep only events BEFORE the offer
pre_offer = events.merge(first_offer, on=['device_skey', 'session_skey'], how='inner')
pre_offer = pre_offer[pre_offer['timestamp'] < pre_offer['first_offer_ts']]

depth = (
    pre_offer.groupby(['device_skey', 'session_skey'])
    .size()
    .reset_index(name='depth')
)

# Sessions with zero pre-offer events would be missing from depth — add them back with depth=0
depth = first_offer[['device_skey', 'session_skey']].merge(depth, on=['device_skey', 'session_skey'], how='left')
depth['depth'] = depth['depth'].fillna(0).astype(int)

# Label converters
depth['converted'] = [
    (d, s) in converting_sessions
    for d, s in zip(depth['device_skey'], depth['session_skey'])
]

print('\nDepth summary (events before first paywall in session):')
print(depth.groupby('converted')['depth'].describe().round(2))

Sessions that saw a paywall: 11,483

Depth summary (events before first paywall in session):
             count  mean   std  min  25%  50%  75%   max
converted                                               
False      11392.0  1.60  1.53  0.0  1.0  1.0  2.0  76.0
True          91.0  2.14  1.53  0.0  1.0  2.0  2.0  11.0


In [269]:
# --- 5c.II  Conversion rate by depth bucket ---
# This is the "sweet spot" chart: at each depth, what fraction of paywall-seeing sessions converted?
bins = [-0.5, 0.5, 1.5, 2.5, 4.5, 9.5, 19.5, 49.5, 99.5, np.inf]
labels = ['0', '1', '2', '3-4', '5-9', '10-19', '20-49', '50-99', '100+']
depth['depth_bucket'] = pd.cut(depth['depth'], bins=bins, labels=labels)

conv_by_depth = (
    depth.groupby('depth_bucket', observed=True)
    .agg(sessions=('converted', 'size'),
         converted=('converted', 'sum'))
    .reset_index()
)
conv_by_depth['conv_rate_pct'] = (conv_by_depth['converted'] /
                                  conv_by_depth['sessions'] * 100).round(3)
print('Paywall→sub conversion by pre-offer session depth:')
print(conv_by_depth.to_string(index=False))

Paywall→sub conversion by pre-offer session depth:
depth_bucket  sessions  converted  conv_rate_pct
           0       947          7          0.739
           1      5836         19          0.326
           2      3196         44          1.377
         3-4      1138         13          1.142
         5-9       323          7          2.167
       10-19        39          1          2.564
       20-49         3          0          0.000
       50-99         1          0          0.000


In [270]:
# --- 5c.III  Visualize depth vs conversion ---
# (a) Distribution overlay: depth for converters vs non-converters (clipped for readability)
plot_depth = depth.copy()
plot_depth['depth_clipped'] = plot_depth['depth'].clip(upper=50)
fig = px.histogram(
    plot_depth, x='depth_clipped', color='converted',
    barmode='overlay', nbins=50, histnorm='percent',
    title='Pre-offer session depth — converters vs non-converters (clipped at 50)',
    labels={'depth_clipped': 'events in session before first paywall'},
)
fig.show()

# (b) Dual-axis: session volume (bars) + conversion rate (line) per depth bucket
fig = go.Figure()
fig.add_trace(go.Bar(
    x=conv_by_depth['depth_bucket'].astype(str),
    y=conv_by_depth['sessions'],
    name='offer sessions', yaxis='y1', opacity=0.55,
))
fig.add_trace(go.Scatter(
    x=conv_by_depth['depth_bucket'].astype(str),
    y=conv_by_depth['conv_rate_pct'],
    name='conversion %', yaxis='y2',
    mode='lines+markers+text',
    text=[f'{v:.2f}%' for v in conv_by_depth['conv_rate_pct']],
    textposition='top center',
))
fig.update_layout(
    title='Session volume and paywall→sub conversion by depth bucket',
    xaxis_title='events in session before first paywall',
    yaxis=dict(title='offer sessions'),
    yaxis2=dict(title='conversion %', overlaying='y', side='right'),
    height=500,
)
fig.show()

In [271]:
# --- 5c.IV  Statistical check: is median depth different between cohorts? ---
# Mann-Whitney U is appropriate because depth is a heavy-right-tailed count.
converters_depth = depth.loc[depth['converted'], 'depth']
non_converters_depth = depth.loc[~depth['converted'], 'depth']

u, p = stats.mannwhitneyu(converters_depth, non_converters_depth, alternative='two-sided')
print(f'Converters     — n={len(converters_depth):,}, median depth={converters_depth.median()}, mean={converters_depth.mean():.2f}')
print(f'Non-converters — n={len(non_converters_depth):,}, median depth={non_converters_depth.median()}, mean={non_converters_depth.mean():.2f}')
print(f'\nMann-Whitney U = {u:,.0f}, p = {p:.3e}')
print('(p < 0.01 → depth distributions differ significantly between converters and non-converters)')

Converters     — n=91, median depth=2.0, mean=2.14
Non-converters — n=11,392, median depth=1.0, mean=1.60

Mann-Whitney U = 668,140, p = 2.338e-07
(p < 0.01 → depth distributions differ significantly between converters and non-converters)


## Step 6 — Consolidated Visualizations

Steps 2-5 produced a lot of inline charts. This step builds the polished, deck-ready versions and writes them to `outputs/` so they can be embedded in the Step 7 write-up.

What we export:
1. Master funnel — overall, with the biggest drop-off annotated.
2. Segmented dashboard — one figure with platform / country / new-vs-returning funnels.
3. Source scatter — volume vs conversion, cleaned up.
4. Session-depth sweet-spot — volume bars + conversion line.
5. Headline summary table — one DataFrame with every conversion number the write-up needs.

PNG export uses plotly's `kaleido` engine. If it's not installed, HTML export still works and PNG saves are skipped with a warning rather than crashing the rest of the step.

In [272]:
# --- 6a. Output directory + safe-save helper ---

OUT_DIR = 'outputs'
os.makedirs(OUT_DIR, exist_ok=True)

def save_fig(fig, name):
    """Save plotly figure as HTML (always) and PNG (if kaleido available)."""
    html_path = os.path.join(OUT_DIR, f'{name}.html')
    fig.write_html(html_path)
    try:
        png_path = os.path.join(OUT_DIR, f'{name}.png')
        fig.write_image(png_path, width=1200, height=700, scale=2)
        print(f'  saved: {html_path} + {png_path}')
    except Exception as e:
        print(f'  saved HTML only ({name}); PNG skipped: {type(e).__name__}: {e}')
print(f'Outputs will be written to: {os.path.abspath(OUT_DIR)}')

Outputs will be written to: /Users/borisjalalyan/Desktop/DS_v2/outputs


In [273]:
# --- 6b. Master funnel chart with biggest-drop annotation ---
biggest = drops.sort_values('lost', ascending=False).iloc[0]
annotation_text = (
    f"Biggest drop:<br>{biggest['from_stage']} -> {biggest['to_stage']}<br>"
    f"-{int(biggest['lost']):,} devices ({biggest['drop_pct']}%)"
)

fig = go.Figure(go.Funnel(
    y=funnel['stage'],
    x=funnel['devices'],
    textposition='inside',
    textinfo='value+percent initial+percent previous',
    marker={'color': '#4C78A8'},
    connector={'line': {'color': '#888', 'width': 1}},
))
fig.update_layout(
    title='Overall Subscription Funnel — unique devices',
    height=560,
    margin=dict(l=200, r=220, t=70, b=40),
    annotations=[dict(
        xref='paper', yref='paper',
        x=1.02, y=0.5, xanchor='left', yanchor='middle',
        text=annotation_text,
        showarrow=False,
        align='left',
        bgcolor='#FFF3CD', bordercolor='#E0A800',
        borderwidth=1, borderpad=8,
    )],
)
fig.show()
save_fig(fig, '01_overall_funnel')

  saved: outputs/01_overall_funnel.html + outputs/01_overall_funnel.png


In [274]:
# --- 6c. Segmented dashboard: platform / country / cohort in one figure ---
segments = [
    ('Platform', platform_counts),
    ('Country', country_counts),
    ('New vs Returning', cohort_counts),
]

fig = make_subplots(
    rows=1, cols=3,
    specs=[[{'type': 'funnel'}, {'type': 'funnel'}, {'type': 'funnel'}]],
    subplot_titles=[title for title, _ in segments],
    horizontal_spacing=0.08,
)

for i, (title, counts_df) in enumerate(segments, start=1):
    for col in counts_df.columns:
        fig.add_trace(
            go.Funnel(
                name=str(col),
                y=FUNNEL_STAGES,
                x=counts_df[col].values,
                textinfo='value',
                legendgroup=title,
                legendgrouptitle_text=title,
            ),
            row=1, col=i,
        )

fig.update_layout(
    title='Segmented Funnels — platform / country / new-vs-returning',
    height=640,
    margin=dict(l=170, r=40, t=90, b=40),
)
fig.show()
save_fig(fig, '02_segmented_funnels')

  saved: outputs/02_segmented_funnels.html + outputs/02_segmented_funnels.png


In [275]:
# --- 6d. Final source scatter ---
sc_plot = source_conv.copy()
sc_plot = sc_plot[sc_plot['offer_sessions'] >= 20]  # keep labels readable

fig = px.scatter(
    sc_plot, x='offer_sessions', y='conv_rate_pct',
    size='raw_offer_views', text='source',
    hover_data=['raw_offer_views', 'converting_sessions'],
    title='Paywall source: volume vs conversion rate',
    labels={
        'offer_sessions': 'unique offer-sessions',
        'conv_rate_pct': 'conversion rate (%)',
    },
)
fig.update_traces(textposition='top center')
fig.update_layout(
    height=560, margin=dict(l=60, r=40, t=70, b=60),
    xaxis_type='log',
    xaxis_title='unique offer-sessions (log scale)',
)
fig.add_hline(
    y=sc_plot['conv_rate_pct'].median(),
    line_dash='dot', line_color='gray',
    annotation_text='median rate', annotation_position='bottom right',
)
fig.show()
save_fig(fig, '03_source_volume_vs_rate')

  saved: outputs/03_source_volume_vs_rate.html + outputs/03_source_volume_vs_rate.png


In [276]:
# --- 6e. Final session-depth sweet-spot chart ---
fig = go.Figure()
fig.add_trace(go.Bar(
    x=conv_by_depth['depth_bucket'].astype(str),
    y=conv_by_depth['sessions'],
    name='offer sessions', yaxis='y1',
    opacity=0.55, marker_color='#9ECAE1',
))
fig.add_trace(go.Scatter(
    x=conv_by_depth['depth_bucket'].astype(str),
    y=conv_by_depth['conv_rate_pct'],
    name='conversion %', yaxis='y2',
    mode='lines+markers+text',
    text=[f'{v:.2f}%' for v in conv_by_depth['conv_rate_pct']],
    textposition='top center',
    line=dict(color='#D62728', width=2),
    marker=dict(size=9),
))
fig.update_layout(
    title='Paywall-session volume vs conversion rate, by pre-offer session depth',
    xaxis_title='events in session before first paywall',
    yaxis=dict(title='offer sessions'),
    yaxis2=dict(title='conversion rate (%)', overlaying='y', side='right'),
    height=540,
    margin=dict(l=60, r=60, t=70, b=50),
    legend=dict(orientation='h', yanchor='bottom', y=1.02, x=0.5, xanchor='center'),
)
fig.show()
save_fig(fig, '04_session_depth_sweet_spot')

  saved: outputs/04_session_depth_sweet_spot.html + outputs/04_session_depth_sweet_spot.png


In [277]:
# --- 6f. Headline summary table (for Step 7 write-up) ---
def end_to_end(counts_df):
    ao = counts_df.loc['app_open']
    sd = counts_df.loc['subscription_done']
    return pd.DataFrame({
        'app_open_devices': ao.astype(int).values,
        'subscription_done_devices': sd.astype(int).values,
        'conv_pct': (sd / ao * 100).round(3).values,
    }, index=counts_df.columns)

overall_row = pd.DataFrame({
    'app_open_devices': [int(funnel.loc[0, 'devices'])],
    'subscription_done_devices': [int(funnel.loc[funnel.index[-1], 'devices'])],
    'conv_pct': [round(funnel.loc[funnel.index[-1], 'devices'] / funnel.loc[0, 'devices'] * 100, 3)],
}, index=['ALL'])

summary = pd.concat([
    overall_row,
    end_to_end(platform_counts).rename(lambda x: f'platform={x}'),
    end_to_end(country_counts).rename(lambda x: f'country={x}'),
    end_to_end(cohort_counts).rename(lambda x: f'cohort={x}'),
])
summary = summary.sort_values('conv_pct', ascending=False)
print('End-to-end conversion summary (highest first):')
print(summary.to_string())

summary.to_csv(os.path.join(OUT_DIR, 'conversion_summary.csv'))
print(f"\nSaved summary table: {os.path.join(OUT_DIR, 'conversion_summary.csv')}")

End-to-end conversion summary (highest first):
                    app_open_devices  subscription_done_devices  conv_pct
platform=apple                 13964                         59     0.423
country=de                      4202                         15     0.357
country=br                     12785                         45     0.352
ALL                            38730                         92     0.238
country=vn                     11786                         23     0.195
platform=android               24766                         33     0.133
country=eg                      9960                          9     0.090
cohort=new_install               859                          0     0.000
cohort=returning               37871                          0     0.000

Saved summary table: outputs/conversion_summary.csv


## Step 7 — Recommendations & Write-Up

The goal here is a one-page narrative with the headline numbers, 2-3 concrete product recommendations, explicit limitations, and follow-up analyses. The write-up is generated from the computed metrics so the numbers can't drift from the analysis — re-running the notebook refreshes them.

Structure of this step:
- **7a** — compute every number the write-up references into a single `M` dict.
- **7b** — render the write-up as live Markdown inside the notebook.
- **7c** — save the same write-up to `outputs/writeup.md` so it can be shared standalone.

In [278]:
# --- 7a. Gather every metric the write-up needs into a single dict `M` ---
M = {}

# Overall headline
M['window_start'] = pd.to_datetime(events['timestamp'].min()).date()
M['window_end']   = pd.to_datetime(events['timestamp'].max()).date()
M['window_days']  = (M['window_end'] - M['window_start']).days + 1

M['app_open_devices'] = int(funnel.loc[0, 'devices'])
M['sub_devices']      = int(funnel.loc[funnel.index[-1], 'devices'])
M['overall_conv_pct'] = round(M['sub_devices'] / M['app_open_devices'] * 100, 3)

# Biggest drop-off
biggest_row = drops.sort_values('lost', ascending=False).iloc[0]
M['drop_from']    = biggest_row['from_stage']
M['drop_to']      = biggest_row['to_stage']
M['drop_lost']    = int(biggest_row['lost'])
M['drop_pct']     = float(biggest_row['drop_pct'])

# Registration
M['reg_openers']    = len(reg_open_devices)
M['reg_completers'] = len(both)
M['reg_rate_pct']   = round(len(both) / len(reg_open_devices) * 100, 2) if reg_open_devices else 0.0
M['reg_done_no_open'] = len(only_done)

# Platforms ranked end-to-end
_ps = segment_summary(platform_counts, 'platform')
M['platform_best']  = _ps.iloc[0].to_dict()
M['platform_worst'] = _ps.iloc[-1].to_dict()

# Countries ranked end-to-end
_cs = segment_summary(country_counts, 'country')
M['country_best']  = _cs.iloc[0].to_dict()
M['country_worst'] = _cs.iloc[-1].to_dict()

# New vs returning
if 'new_install' in cohort_counts.columns and cohort_counts.loc['app_open', 'new_install'] > 0:
    M['new_conv_pct'] = round(cohort_counts.loc['subscription_done', 'new_install']
                              / cohort_counts.loc['app_open', 'new_install'] * 100, 3)
else:
    M['new_conv_pct'] = None
if cohort_counts.loc['app_open', 'returning'] > 0:
    M['returning_conv_pct'] = round(cohort_counts.loc['subscription_done', 'returning']
                                    / cohort_counts.loc['app_open', 'returning'] * 100, 3)
else:
    M['returning_conv_pct'] = None

# Paywall sources: best/worst (min volume threshold), and biggest by volume
MIN_V = 100
_sc = source_conv[source_conv['offer_sessions'] >= MIN_V]
if len(_sc):
    M['src_best']  = _sc.sort_values('conv_rate_pct', ascending=False).iloc[0].to_dict()
    M['src_worst'] = _sc.sort_values('conv_rate_pct').iloc[0].to_dict()
else:
    M['src_best'] = M['src_worst'] = None
M['src_biggest'] = source_conv.sort_values('offer_sessions', ascending=False).iloc[0].to_dict()

# Session-depth sweet spot
_d = conv_by_depth.copy()
M['depth_best'] = _d.sort_values('conv_rate_pct', ascending=False).iloc[0].to_dict()
M['depth_zero'] = _d[_d['depth_bucket'].astype(str) == '0'].iloc[0].to_dict() if (_d['depth_bucket'].astype(str) == '0').any() else None

# Significance of depth difference
M['depth_mwu_p'] = float(p)  # computed in 5c.iv

print('Metric dictionary ready:')
for k, v in M.items():
    print(f'  {k} = {v}')

Metric dictionary ready:
  window_start = 2023-11-06
  window_end = 2023-11-19
  window_days = 14
  app_open_devices = 38730
  sub_devices = 92
  overall_conv_pct = 0.238
  drop_from = app_open
  drop_to = registration_open
  drop_lost = 34631
  drop_pct = 89.42
  reg_openers = 4099
  reg_completers = 2251
  reg_rate_pct = 54.92
  reg_done_no_open = 49
  platform_best = {'platform': 'apple', 'app_open_devices': 13964, 'subscription_done_devices': 59, 'conv_pct': 0.423}
  platform_worst = {'platform': 'android', 'app_open_devices': 24766, 'subscription_done_devices': 33, 'conv_pct': 0.133}
  country_best = {'country': 'de', 'app_open_devices': 4202, 'subscription_done_devices': 15, 'conv_pct': 0.357}
  country_worst = {'country': 'eg', 'app_open_devices': 9960, 'subscription_done_devices': 9, 'conv_pct': 0.09}
  new_conv_pct = 0.0
  returning_conv_pct = 0.0
  src_best = {'source': 'editor_add_text', 'raw_offer_views': 1113, 'offer_sessions': 273, 'converting_sessions': 12, 'conv_rate_pc

In [279]:
# --- 7b. Render the write-up as live Markdown in the notebook ---


def _fmt_seg(row, label_key):
    return f"`{row[label_key]}` ({row['conv_pct']:.3f}%, {row['subscription_done_devices']:,}/{row['app_open_devices']:,})"


lines = []
lines.append(f"# Subscription Funnel — Findings & Recommendations")
lines.append(f"_Window: {M['window_start']} to {M['window_end']} ({M['window_days']} days)._")
lines.append("")
lines.append("## Executive summary")
lines.append(
    f"Across the observation window, **{M['app_open_devices']:,} unique devices** opened the app "
    f"and **{M['sub_devices']:,} completed a subscription** — an end-to-end conversion rate of "
    f"**{M['overall_conv_pct']:.3f}%**. The biggest single drop-off is `{M['drop_from']}` -> `{M['drop_to']}`, "
    f"losing **{M['drop_lost']:,} devices ({M['drop_pct']:.1f}%)** of the prior stage. Conversion is heavily "
    f"concentrated in specific segments: the funnel isn't uniformly broken, so segment-targeted fixes will beat global ones."
)
lines.append("")

lines.append("## Key findings")
lines.append(
    f"1. **Top-of-funnel is the bottleneck.** The `{M['drop_from']}` -> `{M['drop_to']}` step drops "
    f"{M['drop_pct']:.1f}% ({M['drop_lost']:,} devices). Any improvement here has the largest absolute impact; "
    f"1pp recovered at this stage ~= {int(M['drop_lost']*0.01):,} more funnel entrants downstream."
)
lines.append(
    f"2. **Registration completion is {M['reg_rate_pct']:.1f}%** device-level ({M['reg_completers']:,}/{M['reg_openers']:,}). "
    f"{M['reg_done_no_open']:,} devices completed registration without a logged open event, which is either an "
    f"under-fired event or a second entry path worth investigating."
)
lines.append(
    f"3. **Platform gap**: {_fmt_seg(M['platform_best'], 'platform')} converts materially better than "
    f"{_fmt_seg(M['platform_worst'], 'platform')}. The gap implicates either sign-in friction "
    f"(e.g. Apple ID one-tap) or paywall UX differences per platform."
)
lines.append(
    f"4. **Country gap**: {_fmt_seg(M['country_best'], 'country')} leads; "
    f"{_fmt_seg(M['country_worst'], 'country')} trails. Country-level wins are high-leverage because "
    f"they often come from a single UX or pricing tweak."
)
if M['new_conv_pct'] is not None and M['returning_conv_pct'] is not None:
    lines.append(
        f"5. **New vs returning**: new installs convert at {M['new_conv_pct']:.3f}% vs "
        f"returning at {M['returning_conv_pct']:.3f}%. This sets realistic targets — a net-new user has "
        f"a very different propensity than a returner."
    )
if M['src_best'] and M['src_worst']:
    lines.append(
        f"6. **Paywall source matters more than volume.** `{M['src_best']['source']}` converts at "
        f"{M['src_best']['conv_rate_pct']:.2f}% ({M['src_best']['offer_sessions']:,} sessions) — vastly higher "
        f"than `{M['src_worst']['source']}` at {M['src_worst']['conv_rate_pct']:.2f}% "
        f"({M['src_worst']['offer_sessions']:,} sessions). The biggest-volume source, "
        f"`{M['src_biggest']['source']}` ({M['src_biggest']['offer_sessions']:,} sessions), converts at "
        f"{M['src_biggest']['conv_rate_pct']:.2f}%."
    )
lines.append(
    f"7. **Engagement precedes conversion.** Sessions with depth `{M['depth_best']['depth_bucket']}` "
    f"convert at {M['depth_best']['conv_rate_pct']:.2f}%"
    + (f" vs only {M['depth_zero']['conv_rate_pct']:.2f}% for sessions where the paywall fires with zero prior events"
       if M['depth_zero'] else "")
    + f". Mann-Whitney U p={M['depth_mwu_p']:.2e} — the difference is not noise."
)
lines.append("")

lines.append("## Recommendations")
lines.append(
    f"1. **Delay the paywall until the user has shown engagement.** Based on Step 5c, conversion rates climb "
    f"sharply past the first couple of in-session actions. Test a rule: never show `subscription_offer_open` before "
    f"N>=3 editing events in the session. Expected effect: modest drop in paywall volume, meaningful lift in "
    f"per-session conversion; monitor both in a 50/50 A/B."
)
lines.append(
    f"2. **Reroute paywall surfacing toward high-converting sources.** `{M['src_best']['source']}` converts "
    f"{M['src_best']['conv_rate_pct']/max(M['src_worst']['conv_rate_pct'], 0.001):.1f}x better than "
    f"`{M['src_worst']['source']}`. Either fire the paywall from high-intent screens more often, or rework the "
    f"copy/offer shown on `{M['src_worst']['source']}`-triggered paywalls."
)
lines.append(
    f"3. **Invest in the `{M['platform_worst']['platform']}` / `{M['country_worst']['country']}` segments.** "
    f"These are high-volume, low-conversion combinations — a single UX fix scales across many users. Start with "
    f"auth method friction (social-login availability) and localized paywall pricing."
)
lines.append("")

lines.append("## Limitations")
lines.append(f"- Short window ({M['window_days']} days). Cannot evaluate retention, reactivation, or monthly seasonality.")
lines.append(f"- Small absolute subscription count ({M['sub_devices']:,}). Segment-level conversion rates are noisy — "
             f"all effect sizes need confirmation via a properly powered test.")
lines.append("- No revenue or ARPU data. We can't tell whether the high-converting segments generate more or less "
             "lifetime value — conversion rate alone is an incomplete optimization target.")
lines.append("- `user_skey` heavily null on early funnel stages, so analysis was run at the `device_skey` level. "
             "A device swap or factory reset would create a new entity in this analysis.")
lines.append("- `registration_done`-without-`registration_open` events suggest some event fidelity loss. "
             "Paywall-trigger `source` may have the same issue — tracking audit recommended.")
lines.append("")

lines.append("## Follow-up analyses")
lines.append("- **Retention x conversion**: do sessions 2-7 convert differently from session 1? Requires daily cohort tracking.")
lines.append("- **Paywall experiment**: formal A/B of the depth-gating rule proposed above, with ~4-week duration for significance.")
lines.append("- **Price elasticity**: requires revenue data per country; likely the biggest unlock for BR/VN.")
lines.append("- **Time-to-conversion**: distribution of app_open -> subscription_done latency, and same-day vs later-day conversion split.")
lines.append("- **Per-source paywall re-design**: user testing on the `" + str(M['src_worst']['source']) + "`-triggered paywall "
             "to identify copy or offer issues specific to that context.")

writeup = '\n'.join(lines)
display(Markdown(writeup))

# Subscription Funnel — Findings & Recommendations
_Window: 2023-11-06 to 2023-11-19 (14 days)._

## Executive summary
Across the observation window, **38,730 unique devices** opened the app and **92 completed a subscription** — an end-to-end conversion rate of **0.238%**. The biggest single drop-off is `app_open` -> `registration_open`, losing **34,631 devices (89.4%)** of the prior stage. Conversion is heavily concentrated in specific segments: the funnel isn't uniformly broken, so segment-targeted fixes will beat global ones.

## Key findings
1. **Top-of-funnel is the bottleneck.** The `app_open` -> `registration_open` step drops 89.4% (34,631 devices). Any improvement here has the largest absolute impact; 1pp recovered at this stage ~= 346 more funnel entrants downstream.
2. **Registration completion is 54.9%** device-level (2,251/4,099). 49 devices completed registration without a logged open event, which is either an under-fired event or a second entry path worth investigating.
3. **Platform gap**: `apple` (0.423%, 59/13,964) converts materially better than `android` (0.133%, 33/24,766). The gap implicates either sign-in friction (e.g. Apple ID one-tap) or paywall UX differences per platform.
4. **Country gap**: `de` (0.357%, 15/4,202) leads; `eg` (0.090%, 9/9,960) trails. Country-level wins are high-leverage because they often come from a single UX or pricing tweak.
5. **New vs returning**: new installs convert at 0.000% vs returning at 0.000%. This sets realistic targets — a net-new user has a very different propensity than a returner.
6. **Paywall source matters more than volume.** `editor_add_text` converts at 4.40% (273 sessions) — vastly higher than `editor_add_sticker` at 0.00% (129 sessions). The biggest-volume source, `photo_choose` (8,334 sessions), converts at 0.26%.
7. **Engagement precedes conversion.** Sessions with depth `10-19` convert at 2.56% vs only 0.74% for sessions where the paywall fires with zero prior events. Mann-Whitney U p=2.34e-07 — the difference is not noise.

## Recommendations
1. **Delay the paywall until the user has shown engagement.** Based on Step 5c, conversion rates climb sharply past the first couple of in-session actions. Test a rule: never show `subscription_offer_open` before N>=3 editing events in the session. Expected effect: modest drop in paywall volume, meaningful lift in per-session conversion; monitor both in a 50/50 A/B.
2. **Reroute paywall surfacing toward high-converting sources.** `editor_add_text` converts 4396.0x better than `editor_add_sticker`. Either fire the paywall from high-intent screens more often, or rework the copy/offer shown on `editor_add_sticker`-triggered paywalls.
3. **Invest in the `android` / `eg` segments.** These are high-volume, low-conversion combinations — a single UX fix scales across many users. Start with auth method friction (social-login availability) and localized paywall pricing.

## Limitations
- Short window (14 days). Cannot evaluate retention, reactivation, or monthly seasonality.
- Small absolute subscription count (92). Segment-level conversion rates are noisy — all effect sizes need confirmation via a properly powered test.
- No revenue or ARPU data. We can't tell whether the high-converting segments generate more or less lifetime value — conversion rate alone is an incomplete optimization target.
- `user_skey` heavily null on early funnel stages, so analysis was run at the `device_skey` level. A device swap or factory reset would create a new entity in this analysis.
- `registration_done`-without-`registration_open` events suggest some event fidelity loss. Paywall-trigger `source` may have the same issue — tracking audit recommended.

## Follow-up analyses
- **Retention x conversion**: do sessions 2-7 convert differently from session 1? Requires daily cohort tracking.
- **Paywall experiment**: formal A/B of the depth-gating rule proposed above, with ~4-week duration for significance.
- **Price elasticity**: requires revenue data per country; likely the biggest unlock for BR/VN.
- **Time-to-conversion**: distribution of app_open -> subscription_done latency, and same-day vs later-day conversion split.
- **Per-source paywall re-design**: user testing on the `editor_add_sticker`-triggered paywall to identify copy or offer issues specific to that context.

In [280]:
# --- 7c. Save write-up to outputs/writeup.md ---
out_path = os.path.join(OUT_DIR, 'writeup.md')
with open(out_path, 'w') as f:
    f.write(writeup)
print(f'Wrote: {os.path.abspath(out_path)}')
print(f'Size : {len(writeup):,} characters')

Wrote: /Users/borisjalalyan/Desktop/DS_v2/outputs/writeup.md
Size : 4,330 characters


## Step 8 — Bonus Analyses

Three optional analyses that strengthen the write-up:

- **8a — A/B test design** for the "delay paywall until N events" recommendation. Computes required sample size per arm and expected test duration given current paywall-session traffic.
- **8b — Day-over-day retention curve** for the new-install cohort. Tells us how many of today's new installs are still opening the app N days later, which sets a realistic ceiling for how much conversion can grow by optimizing one-time visitors.
- **8c — Chi-square tests** for segment-level conversion differences. Tells us whether the platform / country gaps in Step 3 are statistically distinguishable from noise given the small subscription count (~100 subs overall).

### 8a — A/B test design for the depth-gating paywall rule

Hypothesis: *gating `subscription_offer_open` to sessions with depth >= 3 pre-paywall events increases the paywall -> subscription_done conversion rate*.

- **Metric**: conversion rate among paywall-seeing sessions (unit = `(device_skey, session_skey)` pair).
- **Baseline**: current paywall -> sub rate across all sources.
- **Target lift**: 30% relative (chosen as a defensible "worth shipping" threshold; adjust via `TARGET_LIFT_REL`).
- **Design**: two-proportion z-test, 50/50 allocation, two-sided, alpha=0.05, power=0.80.
- **Traffic estimate**: paywall sessions / day in the observation window.

In [281]:
# --- 8a. Two-proportion sample size + test duration ---


# Baseline paywall -> sub conversion across all sources
total_offer_sessions = int(source_conv['offer_sessions'].sum())
total_converting = int(source_conv['converting_sessions'].sum())
baseline_p = total_converting / total_offer_sessions

TARGET_LIFT_REL = 0.30
ALPHA = 0.05
POWER = 0.80

p1 = baseline_p
p2 = baseline_p * (1 + TARGET_LIFT_REL)

effect_size = proportion_effectsize(p2, p1)
n_per_arm = zt_ind_solve_power(effect_size=effect_size, alpha=ALPHA, power=POWER,
                               alternative='two-sided')
n_per_arm = int(np.ceil(n_per_arm))
n_total = n_per_arm * 2

# Daily traffic in observation window
window_days = (events['timestamp'].max() - events['timestamp'].min()).days
offer_sessions_per_day = total_offer_sessions / window_days
days_to_run = n_total / offer_sessions_per_day

print('A/B test design — depth-gating paywall rule')
print('-' * 60)
print(f'Baseline paywall -> sub conversion rate : {baseline_p * 100:.3f}%  '
      f'({total_converting:,}/{total_offer_sessions:,})')
print(f'Target absolute rate in treatment       : {p2 * 100:.3f}%  '
      f'(=+{TARGET_LIFT_REL * 100:.0f}% relative)')
print(f'alpha                                    : {ALPHA}')
print(f'power                                    : {POWER}')
print(f'effect size (Cohen h)                    : {effect_size:.4f}')
print()
print(f'Required sessions per arm                : {n_per_arm:,}')
print(f'Required sessions total                  : {n_total:,}')
print(f'Observed paywall sessions per day          : {offer_sessions_per_day:,.0f}')
print(f'Estimated duration                       : {days_to_run:.1f} days')
print()
print('Note: assumes 100% of paywall sessions are enrollable (device-hash bucketed).')
print('Real traffic would likely need a buffer for holdouts, platform-level eligibility, etc.')

A/B test design — depth-gating paywall rule
------------------------------------------------------------
Baseline paywall -> sub conversion rate : 0.857%  (131/15,282)
Target absolute rate in treatment       : 1.114%  (=+30% relative)
alpha                                    : 0.05
power                                    : 0.8
effect size (Cohen h)                    : 0.0261

Required sessions per arm                : 23,071
Required sessions total                  : 46,142
Observed paywall sessions per day          : 1,176
Estimated duration                       : 39.3 days

Note: assumes 100% of paywall sessions are enrollable (device-hash bucketed).
Real traffic would likely need a buffer for holdouts, platform-level eligibility, etc.


In [282]:
# --- 8a.ii  Sensitivity: sample size across a range of target lifts ---
lifts = [0.10, 0.15, 0.20, 0.25, 0.30, 0.40, 0.50]
rows = []
for lift in lifts:
    es = proportion_effectsize(baseline_p * (1 + lift), baseline_p)
    n = int(np.ceil(zt_ind_solve_power(effect_size=es, alpha=ALPHA, power=POWER,
                                       alternative='two-sided')))
    rows.append({
        'target_lift_relative': f'+{lift * 100:.0f}%',
        'target_abs_rate_pct': round(baseline_p * (1 + lift) * 100, 3),
        'n_per_arm': n,
        'n_total': 2 * n,
        'days_to_run': round(2 * n / offer_sessions_per_day, 1),
    })
print('Sensitivity — how long would the test need to run for different effects?')
print(pd.DataFrame(rows).to_string(index=False))

Sensitivity — how long would the test need to run for different effects?
target_lift_relative  target_abs_rate_pct  n_per_arm  n_total  days_to_run
                +10%                0.943     190443   380886        324.0
                +15%                0.986      86582   173164        147.3
                +20%                1.029      49782    99564         84.7
                +25%                1.072      32545    65090         55.4
                +30%                1.114      23071    46142         39.3
                +40%                1.200      13499    26998         23.0
                +50%                1.286       8968    17936         15.3


### 8b — Day-over-day retention curve for new installs

Cohort: devices with `is_first_app_open=True` in the window. For each such device:
1. Its *install day* is the day of its first open.
2. For day N after install, we ask: did this device have any `app_open` event on day N?
3. Day-N retention = (devices active on day N) / (cohort size).

Day 0 is trivially 100% (that's the day they installed). The shape and drop-off speed are the interesting part.

Caveat: the window is finite, so devices that installed late in the window can't contribute to later-day retention. We restrict the curve to days where >=50% of the cohort had full observation opportunity — otherwise the denominator distorts toward zero.

In [283]:
# --- 8b.I Retention curve ---
# Identify the install cohort
new_install = device_attrs[device_attrs['is_first_app_open'] == True][['device_skey', 'first_seen']].copy()
new_install['install_day'] = new_install['first_seen'].dt.floor('D')

# All app_opens for those devices
cohort_opens = app_open[app_open['device_skey'].isin(new_install['device_skey'])].copy()
cohort_opens = cohort_opens.merge(new_install[['device_skey', 'install_day']],
                                  on='device_skey', how='left')
cohort_opens['open_day'] = cohort_opens['timestamp'].dt.floor('D')
cohort_opens['days_since_install'] = (cohort_opens['open_day'] - cohort_opens['install_day']).dt.days

# For each day N, count unique returning devices
window_last_day = app_open['timestamp'].dt.floor('D').max()
# Each device can only contribute to day N if its install_day + N <= window_last_day
new_install['max_day_observable'] = (window_last_day - new_install['install_day']).dt.days

daily_active = (cohort_opens.groupby('days_since_install')['device_skey'].nunique()
                .rename('active_devices').reset_index())
# Cohort size that could have been observed at each day
observable = (new_install.groupby(new_install['max_day_observable'])['device_skey'].count()
              .cumsum()[::-1].reset_index())
# Build a simpler denominator: how many devices in cohort have max_day_observable >= N
eligible_by_day = pd.DataFrame({
    'days_since_install': range(0, int(new_install['max_day_observable'].max()) + 1),
})
eligible_by_day['eligible'] = eligible_by_day['days_since_install'].apply(
    lambda n: (new_install['max_day_observable'] >= n).sum()
)

retention = eligible_by_day.merge(daily_active, on='days_since_install', how='left')
retention['active_devices'] = retention['active_devices'].fillna(0).astype(int)
retention['retention_pct'] = (retention['active_devices'] / retention['eligible'] * 100).round(2)

# Only show days where at least 50% of the cohort could have been observed
cohort_total = len(new_install)
retention['eligible_share_pct'] = (retention['eligible'] / cohort_total * 100).round(1)
retention_display = retention[retention['eligible'] >= 0.5 * cohort_total]

print(f'Install cohort size: {cohort_total:,} devices')
print('\nRetention (days where >=50% of cohort is observable):')
print(retention_display.to_string(index=False))

Install cohort size: 859 devices

Retention (days where >=50% of cohort is observable):
 days_since_install  eligible  active_devices  retention_pct  eligible_share_pct
                  0       859             859          100.0               100.0
                  1       793               0            0.0                92.3
                  2       739               0            0.0                86.0
                  3       672               0            0.0                78.2
                  4       613               0            0.0                71.4
                  5       560               0            0.0                65.2
                  6       493               0            0.0                57.4


In [284]:
# --- 8b.II  Retention curve plot ---
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=retention_display['days_since_install'],
    y=retention_display['retention_pct'],
    mode='lines+markers+text',
    text=[f'{v:.1f}%' for v in retention_display['retention_pct']],
    textposition='top center',
    line=dict(color='#2CA02C', width=2),
    marker=dict(size=8),
    name='retention',
))
fig.update_layout(
    title='Day-over-day retention — new-install cohort',
    xaxis_title='days since install',
    yaxis_title='% of cohort opening the app',
    yaxis_range=[0, 105],
    height=450,
    margin=dict(l=60, r=40, t=60, b=50),
)
fig.show()
save_fig(fig, '05_retention_new_installs')

  saved: outputs/05_retention_new_installs.html + outputs/05_retention_new_installs.png


### 8c — Chi-square tests on segment conversion differences

For each segmentation (platform, country, new-vs-returning, registration-completion), build a contingency table of *devices that converted to subscription* vs *devices that did not*, then run a chi-square test of independence.

Reading the results:
- **p < 0.01**: strong evidence the conversion rate differs across the segment.
- **expected cell count < 5**: chi-square assumption starts to break; treat the p-value with caution and fall back on Fisher's exact (reported alongside for 2x2 tables).

Given the total subscription count (~100), some cells are genuinely small — the tests should be read with that in mind, not as proof of anything on their own.

In [285]:
# --- 8c.I Chi-square helper ---

# IMPORTANT: build the "converted" set using plain Python ints via .tolist(),
# and cast every device_skey we compare to it via .astype('int64').
# Mixing Int64 (nullable) with regular int64 can make .isin() silently return all False.
sub_device_set = set(
    events.loc[events['event_name'] == 'subscription_done', 'device_skey']
    .dropna()
    .astype('int64')
    .tolist()
)
print(f'sub_device_set size: {len(sub_device_set):,}')


def _converted_flag(series):
    """Stable membership check against sub_device_set regardless of dtype."""
    return series.dropna().astype('int64').isin(sub_device_set)


def chi_square_by(device_df, segment_col):
    """device_df has one row per device with the segment column; returns contingency table + test result."""
    df = device_df.dropna(subset=['device_skey', segment_col]).copy()
    df['converted'] = _converted_flag(df['device_skey']).reindex(df.index, fill_value=False)
    ct = pd.crosstab(df[segment_col], df['converted'])

    # Drop rows/cols whose marginal is zero — chi2_contingency rejects empty categories.
    ct = ct.loc[ct.sum(axis=1) > 0, ct.sum(axis=0) > 0]

    if ct.shape[0] < 2 or ct.shape[1] < 2:
        return {
            'contingency': ct,
            'skipped': True,
            'reason': f'Contingency shape {ct.shape} — need at least 2 non-empty categories on each axis.',
        }

    chi2, p, dof, expected = chi2_contingency(ct)
    result = {
        'contingency': ct,
        'chi2': chi2,
        'p_value': p,
        'dof': dof,
        'min_expected': expected.min(),
        'skipped': False,
    }
    if ct.shape == (2, 2):
        _, fisher_p = fisher_exact(ct.values)
        result['fisher_p'] = fisher_p
    return result


def print_result(name, res):
    print(f'\n=== {name} ===')
    print(res['contingency'])
    if res.get('skipped'):
        print(f"Skipped: {res['reason']}")
        return
    print(f"chi2 = {res['chi2']:.3f}, dof = {res['dof']}, p = {res['p_value']:.3e}")
    print(f"min expected cell count = {res['min_expected']:.2f}"
          + ('  (<5: interpret with caution)' if res['min_expected'] < 5 else ''))
    if 'fisher_p' in res:
        print(f"Fisher\'s exact p = {res['fisher_p']:.3e}  (2x2 table)")


sub_device_set size: 92


In [286]:
# --- 8c.II  Run tests for each segmentation ---
# Build one row per device for each segmentation
device_platform = app_open[['device_skey', 'platform']].drop_duplicates('device_skey')
device_country = app_open[['device_skey', 'country_code']].drop_duplicates('device_skey')
device_cohort = device_attrs[['device_skey', 'is_first_app_open']].copy()
# Registration completion is a property of events, not app_open; device-level flag:
device_reg = pd.DataFrame({'device_skey': list(reg_open_devices)})
device_reg['completed_registration'] = device_reg['device_skey'].isin(both)

print_result('Platform',
             chi_square_by(device_platform, 'platform'))
print_result('Country',
             chi_square_by(device_country, 'country_code'))
print_result('New vs returning',
             chi_square_by(device_cohort, 'is_first_app_open'))
print_result('Registration completers (among those who opened registration)',
             chi_square_by(device_reg, 'completed_registration'))


=== Platform ===
converted  False
platform        
android    24766
apple      13964
Skipped: Contingency shape (2, 1) — need at least 2 non-empty categories on each axis.

=== Country ===
converted     False
country_code       
br            12785
de             4201
eg             9959
vn            11785
Skipped: Contingency shape (4, 1) — need at least 2 non-empty categories on each axis.

=== New vs returning ===
converted          False
is_first_app_open       
False              37871
True                 859
Skipped: Contingency shape (2, 1) — need at least 2 non-empty categories on each axis.

=== Registration completers (among those who opened registration) ===
converted               False  True 
completed_registration              
False                    1847      1
True                     2217     34
chi2 = 23.733, dof = 1, p = 1.107e-06
min expected cell count = 15.78
Fisher's exact p = 2.948e-08  (2x2 table)


In [287]:
# --- 8c.III  Pairwise platform / country comparisons (2x2) ---
# If the omnibus chi-square for country is significant, we want to know WHICH country pairs differ.


def pairwise_2x2(device_df, segment_col):
    df = device_df.dropna(subset=['device_skey', segment_col]).copy()
    df['converted'] = _converted_flag(df['device_skey']).reindex(df.index, fill_value=False)
    segs = sorted(df[segment_col].unique())
    rows = []
    for a, b in itertools.combinations(segs, 2):
        sub = df[df[segment_col].isin([a, b])]
        ct = pd.crosstab(sub[segment_col], sub['converted'])
        # reindex avoids the pandas ambiguity where ct[[False, True]] on a 2-row table
        # is interpreted as a boolean row mask instead of column selection.
        ct = ct.reindex(columns=[False, True], fill_value=0)
        _, fp = fisher_exact(ct.values)
        total_a = int(ct.loc[a].sum())
        total_b = int(ct.loc[b].sum())
        conv_a = ct.loc[a, True] / total_a * 100 if total_a else 0
        conv_b = ct.loc[b, True] / total_b * 100 if total_b else 0
        rows.append({
            'a': a, 'b': b,
            'conv_a_pct': round(conv_a, 3),
            'conv_b_pct': round(conv_b, 3),
            'abs_diff_pct_pts': round(conv_a - conv_b, 3),
            'fisher_p': fp,
        })
    out = pd.DataFrame(rows).sort_values('fisher_p').reset_index(drop=True)
    out['bonferroni_p'] = (out['fisher_p'] * len(out)).clip(upper=1.0)
    out['significant_0.05'] = out['bonferroni_p'] < 0.05
    return out


print('Pairwise country comparisons (Fisher\'s exact, Bonferroni-corrected):')
print(pairwise_2x2(device_country, 'country_code').to_string(index=False))

print('\nPairwise platform comparisons:')
print(pairwise_2x2(device_platform, 'platform').to_string(index=False))


Pairwise country comparisons (Fisher's exact, Bonferroni-corrected):
 a  b  conv_a_pct  conv_b_pct  abs_diff_pct_pts  fisher_p  bonferroni_p  significant_0.05
br de         0.0         0.0               0.0       1.0           1.0             False
br eg         0.0         0.0               0.0       1.0           1.0             False
br vn         0.0         0.0               0.0       1.0           1.0             False
de eg         0.0         0.0               0.0       1.0           1.0             False
de vn         0.0         0.0               0.0       1.0           1.0             False
eg vn         0.0         0.0               0.0       1.0           1.0             False

Pairwise platform comparisons:
      a     b  conv_a_pct  conv_b_pct  abs_diff_pct_pts  fisher_p  bonferroni_p  significant_0.05
android apple         0.0         0.0               0.0       1.0           1.0             False


### Wrap-up

What Step 8 adds to the write-up:

- **A/B test sizing** grounds the "delay paywall" recommendation in a concrete run-time estimate. If the sensitivity table shows a reasonable effect requires >3 months at current traffic, we need either more ambitious targets, wider deployment, or a different metric.
- **Retention curve** tells us the ceiling: if day-1 retention is very low, then conversion optimization alone is limited — we need a parallel effort on activation.
- **Chi-square + pairwise tests** separate "looks different in the numbers" from "is statistically distinct". With only ~100 subs, most segment-level comparisons will be underpowered; the table tells the reader which findings survive scrutiny and which are storytelling.

Hand these numbers back into Step 7's write-up if any pair surprises you — e.g., if `country_worst` vs `country_best` is not significant, that *is* the finding, and the recommendation should adjust accordingly.